# 🚗 Path 2 — YOLO11 + ByteTrack (Physics-Calibrated v3)
**All 5 issues fixed. Scale-invariant detection via height-normalized velocity.**

| Issue | Old | Fixed |
|---|---|---|
| Scale sensitivity | Absolute px/s² (near ≠ far) | **BH/s² (bbox-height normalized)** |
| Pedestrian detection | Speed drop only | **Overlap + decel OR ped speed spike** |
| Single vehicle | Percentage threshold | **BH/s² + aspect ratio change** |
| Low-speed false flags | 10km/h stop = 89% drop flag | **BH/s² — slow stops produce ~1 BH/s²** |
| FPS fairness | px/s (varies by FPS) | **Time-based velocity computation** |

> **Runtime:** T4 GPU

In [1]:
# 1.1 Install
!pip install -q ultralytics lapx decord
print("Restart runtime, then run from 1.2")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 109.1 MB/s eta 0:00:00
Restart runtime, then run from 1.2


In [2]:
# 1.2 Mount + imports
from google.colab import drive
import os
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print("Drive already mounted")

import json, time, math, random
from pathlib import Path
from collections import defaultdict, deque
import cv2, numpy as np, torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm import tqdm
from ultralytics import YOLO

try:
    import decord
    decord.bridge.set_bridge('native')
    USE_DECORD = True
except ImportError:
    USE_DECORD = False

BASE        = Path('/content/drive/MyDrive/Car_accident_detection')
VIDEOS_DIR  = BASE / 'Dataset/forth_investigation'
FRAMES_DIR  = BASE / 'Dataset/manual/extracted_frames'
WORK_DIR    = Path('/content/drive/MyDrive/Colab Notebooks/grad-project/program-memory')
RESULTS_DIR = WORK_DIR / 'tracking_results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
VEHICLE_CLS = {2:'car',3:'motorcycle',5:'bus',7:'truck',1:'bicycle'}
VEHICLE_IDS = set(VEHICLE_CLS.keys())

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Mounted at /content/drive
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
GPU: Tesla T4


In [3]:
# 2.1 Load YOLO11n
try:    model = YOLO('yolo11n.pt')
except: model = YOLO('yolov8n.pt'); print("Using yolov8n")
model.to(DEVICE)
print(f"YOLO loaded | VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

YOLO loaded | VRAM: 0.01 GB


In [4]:
# 3.1 Height-normalized VehicleTracker (FIX #3 — scale invariance)
def box_aspect(b):
    w=max(1,b[2]-b[0]); h=max(1,b[3]-b[1]); return w/h

def box_height(b):
    """Bbox height in pixels — used as spatial scale reference."""
    return max(1, b[3]-b[1])

def center_dist(b1,b2):
    c1=((b1[0]+b1[2])/2,(b1[1]+b1[3])/2)
    c2=((b2[0]+b2[2])/2,(b2[1]+b2[3])/2)
    return math.hypot(c1[0]-c2[0],c1[1]-c2[1])

def box_iou(b1,b2):
    ix1=max(b1[0],b2[0]); iy1=max(b1[1],b2[1])
    ix2=min(b1[2],b2[2]); iy2=min(b1[3],b2[3])
    if ix2<=ix1 or iy2<=iy1: return 0.0
    inter=(ix2-ix1)*(iy2-iy1)
    a1=(b1[2]-b1[0])*(b1[3]-b1[1]); a2=(b2[2]-b2[0])*(b2[3]-b2[1])
    return inter/(a1+a2-inter+1e-6)


class VehicleTracker:
    """
    Tracks vehicles using YOLO11 + ByteTrack.

    Speed is stored in TWO units:
      speed_pps   : absolute pixels/sec (for visualization)
      speed_bhps  : bbox-heights/sec (scale-invariant — FIX #3)

    Why bbox-height normalization?
      A car 10m away has bbox_height=80px, moves 300 px/s → 3.75 BH/s
      A car 40m away has bbox_height=20px, moves 75 px/s  → 3.75 BH/s
      Same real-world speed → same normalized speed. Physics is preserved.
    """
    def __init__(self, fps: float, history_len: int = 8):
        self.fps         = fps
        self.hist_len    = history_len
        # per track: deque of (frame_num, cx, cy, bbox_height)
        self.pos_history = defaultdict(lambda: deque(maxlen=history_len))
        self.bh_history  = defaultdict(lambda: deque(maxlen=history_len))
        self.aspect_hist = defaultdict(lambda: deque(maxlen=history_len))

    def update(self, frame_bgr: np.ndarray, frame_num: int):
        results = model.track(frame_bgr, tracker='bytetrack.yaml',
                               classes=list(VEHICLE_IDS), persist=True, verbose=False)
        r = results[0]
        vehicles = []; pedestrians = []
        if r.boxes is None or r.boxes.id is None:
            return vehicles, pedestrians
        for box, tid_t, cls_t, conf_t in zip(
                r.boxes.xyxy, r.boxes.id, r.boxes.cls, r.boxes.conf):
            tid    = int(tid_t.item())
            cls_id = int(cls_t.item())
            x1,y1,x2,y2 = [int(v) for v in box.tolist()]
            cx,cy = (x1+x2)/2, (y1+y2)/2
            bh    = box_height([x1,y1,x2,y2])
            self.pos_history[tid].append((frame_num, cx, cy))
            self.bh_history[tid].append(bh)
            self.aspect_hist[tid].append(box_aspect([x1,y1,x2,y2]))
            # Compute velocity
            h = list(self.pos_history[tid])
            vx_pps=vy_pps=0.0
            if len(h)>=2:
                f0,x0,y0=h[0]; f1,x1h,y1h=h[-1]
                dt=(f1-f0)/self.fps
                if dt>0: vx_pps=(x1h-x0)/dt; vy_pps=(y1h-y0)/dt
            speed_pps  = math.hypot(vx_pps,vy_pps)
            speed_bhps = speed_pps / bh   # bbox-heights per second

            entry = {'id':tid,'class':VEHICLE_CLS.get(cls_id,'vehicle'),
                      'bbox':[x1,y1,x2,y2],'cx':cx,'cy':cy,
                      'speed_pps':round(speed_pps,2),
                      'speed_bhps':round(speed_bhps,3),  # scale-invariant
                      'vx_pps':round(vx_pps,2),'vy_pps':round(vy_pps,2),
                      'conf':round(float(conf_t),3),
                      'bbox_height':bh}
            if cls_id == 0: pedestrians.append(entry)
            else: vehicles.append(entry)
        return vehicles, pedestrians

    def decel_bh_s2(self, tid, window_frames=4):
        """
        Deceleration in bbox-heights/s² (scale-invariant).
        Normal hard brake : ~1-2 BH/s²
        Crash impact      : >4-5 BH/s²
        """
        # Reconstruct BH-normalized speed history
        ph = list(self.pos_history[tid])
        bh_h = list(self.bh_history[tid])
        if len(ph) < window_frames+1 or len(bh_h) < window_frames+1:
            return 0.0
        # speed now (in BH/s)
        f1,x1h,y1h=ph[-1]; f0,x0h,y0h=ph[-window_frames]
        dt=(f1-f0)/self.fps
        if dt<=0: return 0.0
        speed_now  = math.hypot(x1h-x0h,y1h-y0h)/dt / max(1,bh_h[-1])
        # speed before (using earlier window)
        if len(ph)<window_frames*2+1: return 0.0
        f3,x3,y3=ph[-window_frames-1]; f2,x2h,y2h=ph[-window_frames*2]
        dt2=(f3-f2)/self.fps
        if dt2<=0: return 0.0
        speed_prev = math.hypot(x3-x2h,y3-y2h)/dt2 / max(1,bh_h[-window_frames-1])
        # decel = (prev_speed - current_speed) / dt
        return max(0.0, (speed_prev - speed_now) / ((window_frames/self.fps)))

    def aspect_change(self, tid):
        h=list(self.aspect_hist[tid])
        if len(h)<4: return 0.0
        return abs(h[-1]-h[0])/max(h[0],1e-3)

    def ped_speed_spike(self, pid, window_frames=3):
        ph=list(self.pos_history[pid])
        bh_h=list(self.bh_history[pid])
        if len(ph)<window_frames+1: return 0.0
        f1,x1h,y1h=ph[-1]; f0,x0h,y0h=ph[-window_frames]
        dt=(f1-f0)/self.fps
        if dt<=0: return 0.0
        speed_now=math.hypot(x1h-x0h,y1h-y0h)/dt/max(1,bh_h[-1])
        if len(ph)<window_frames*2+1: return speed_now
        f3,x3,y3=ph[-window_frames-1]; f2,x2h,y2h=ph[-window_frames*2]
        dt2=(f3-f2)/self.fps
        if dt2<=0: return speed_now
        speed_prev=math.hypot(x3-x2h,y3-y2h)/dt2/max(1,bh_h[-window_frames-1])
        return max(0.0, speed_now-speed_prev)

print("VehicleTracker v3 ready — height-normalized velocities (BH/s, BH/s²)")
print()
print("Why BH/s²:")
print("  Car 10m away: bbox_h=80px, speed=300px/s → 3.75 BH/s")
print("  Car 40m away: bbox_h=20px, speed=75px/s  → 3.75 BH/s")
print("  Same real physics → same number regardless of distance")

VehicleTracker v3 ready — height-normalized velocities (BH/s, BH/s²)

Why BH/s²:
  Car 10m away: bbox_h=80px, speed=300px/s → 3.75 BH/s
  Car 40m away: bbox_h=20px, speed=75px/s  → 3.75 BH/s
  Same real physics → same number regardless of distance


In [5]:
# 3.2 Physics-calibrated collision detector (BH/s² thresholds, FIX #3+#5)
class FullCollisionDetector:
    """
    All thresholds in BH/s² (bbox-height per second squared).

    Calibration (from road safety physics, normalized to vehicle size):
      Normal hard brake  : ~1.5 BH/s²  (0.8g at typical scales)
      Crash impact       : >4.0 BH/s²  (2g+)
      Severe crash       : >8.0 BH/s²
      Threshold = 3.5 BH/s² — safely between braking and impact

    FIX for low-speed false positives:
      A car at 10 km/h stopping: speed_bhps ≈ 0.4 BH/s to begin with.
      decel = 0.4/0.3s ≈ 1.3 BH/s² → well below 3.5 → NOT flagged.
    """
    # Mode A: vehicle-vehicle
    VV_IOU        = 0.05
    VV_DIST_BH    = 2.0    # collision if within 2 bbox-heights (proximity)
    VV_DECEL      = 3.5    # BH/s²
    VV_MIN_SPEED  = 0.5    # BH/s — must have been moving

    # Mode B: single-vehicle (motorcycle fall, flip)
    SV_DECEL      = 4.0    # BH/s² — stricter for solo events
    SV_MIN_SPEED  = 0.5
    SV_ASPECT_CHG = 0.40
    SV_ALONE_BH   = 4.0    # alone if no vehicle within 4 bbox-heights

    # Mode C: pedestrian strike
    PS_IOU        = 0.03
    PS_VEH_DECEL  = 2.5    # BH/s² — vehicle slows after contact
    PS_VEH_MIN_SPEED = 0.4 # BH/s
    PS_PED_SPIKE  = 2.0    # BH/s spike in pedestrian speed

    # Mode D: static impact (lamppost, barrier)
    SI_DECEL      = 5.0    # BH/s² — very high
    SI_MIN_SPEED  = 1.0    # BH/s — was moving meaningfully
    SI_ALONE_BH   = 5.0    # px / bbox_height

    COOLDOWN = 12

    def __init__(self, fps: float, tracker: VehicleTracker):
        self.fps     = fps
        self.tracker = tracker  # shares history
        self.alerts  = {}       # tid -> last alert frame

    def _ok(self, tid, fn):
        return fn - self.alerts.get(tid, -9999) >= self.COOLDOWN

    def _make_ev(self, fn, ctype, inv, all_v, extra=None):
        cbbox=[min(t['bbox'][0] for t in inv),min(t['bbox'][1] for t in inv),
               max(t['bbox'][2] for t in inv),max(t['bbox'][3] for t in inv)]
        # 20% padding for VLM crop (FIX #4)
        pad=0.20
        W=cbbox[2]-cbbox[0]; H=cbbox[3]-cbbox[1]
        # (will be clamped to frame by VLM notebook)
        cbbox_padded=[int(cbbox[0]-W*pad),int(cbbox[1]-H*pad),
                       int(cbbox[2]+W*pad),int(cbbox[3]+H*pad)]
        ev={'frame_num':fn,'time_sec':round(fn/self.fps,2),
             'collision_type':ctype,
             'vehicle_a':inv[0],'vehicle_b':inv[1] if len(inv)>1 else None,
             'collision_bbox':cbbox,'collision_bbox_padded':cbbox_padded,
             'all_tracks':all_v,
             'iou':box_iou(inv[0]['bbox'],inv[1]['bbox']) if len(inv)>1 else 0.0,
             'decel_a_bhs2':round(self.tracker.decel_bh_s2(inv[0]['id']),3),
             'decel_b_bhs2':round(self.tracker.decel_bh_s2(inv[1]['id']),3) if len(inv)>1 else 0.0}
        if extra: ev.update(extra)
        for t in inv: self.alerts[t['id']]=fn
        return ev

    def check(self, vehicles, pedestrians, frame_num):
        all_t=vehicles+pedestrians

        # ── A: vehicle vs vehicle ─────────────────────────
        for i in range(len(vehicles)):
            for j in range(i+1,len(vehicles)):
                a,b=vehicles[i],vehicles[j]
                if not self._ok(a['id'],frame_num): continue
                iou=box_iou(a['bbox'],b['bbox'])
                dist_bh=center_dist(a['bbox'],b['bbox'])/max(a['bbox_height'],b['bbox_height'],1)
                if iou<self.VV_IOU and dist_bh>self.VV_DIST_BH: continue
                # Must have been moving (avoid parked cars)
                ah=list(self.tracker.pos_history[a['id']])
                bh=list(self.tracker.pos_history[b['id']])
                max_sa=a['speed_bhps'] if len(ah)<3 else max(
                    math.hypot(ah[k+1][1]-ah[k][1],ah[k+1][2]-ah[k][2])/(
                        max(1,(ah[k+1][0]-ah[k][0])/self.fps)*a['bbox_height'])
                    for k in range(len(ah)-1)) if ah else 0
                if max(a['speed_bhps'],b['speed_bhps'],max_sa) < self.VV_MIN_SPEED: continue
                da=self.tracker.decel_bh_s2(a['id']); db=self.tracker.decel_bh_s2(b['id'])
                if max(da,db)<self.VV_DECEL: continue
                return self._make_ev(frame_num,'vehicle_vehicle',[a,b],vehicles,
                                      extra={'decel_bhs2':round(max(da,db),3)})

        # ── C: pedestrian strike ─────────────────────────
        for v in vehicles:
            if not self._ok(v['id'],frame_num): continue
            if v['speed_bhps']<self.PS_VEH_MIN_SPEED: continue
            for p in pedestrians:
                if box_iou(v['bbox'],p['bbox'])<self.PS_IOU: continue
                dv=self.tracker.decel_bh_s2(v['id'])
                sig2a=dv>=self.PS_VEH_DECEL
                ped_spk=self.tracker.ped_speed_spike(p['id'])
                sig2b=ped_spk>=self.PS_PED_SPIKE
                if not(sig2a or sig2b): continue
                return self._make_ev(frame_num,'pedestrian_strike',[v,{**p,'speed_pps':0}],
                                      vehicles,
                                      extra={'veh_decel_bhs2':round(dv,3),
                                             'ped_speed_spike_bhs':round(ped_spk,3),
                                             'signal':'2A' if sig2a else '2B'})

        # ── B: single vehicle anomaly ─────────────────────
        for v in vehicles:
            if not self._ok(v['id'],frame_num): continue
            if v['speed_bhps']<self.SV_MIN_SPEED: continue
            dv=self.tracker.decel_bh_s2(v['id'])
            if dv<self.SV_DECEL: continue
            asp=self.tracker.aspect_change(v['id'])
            is_moto=v['class'] in ('motorcycle','bicycle')
            others=[u for u in vehicles if u['id']!=v['id']]
            alone=all(center_dist(v['bbox'],u['bbox'])/v['bbox_height']>self.SV_ALONE_BH
                       for u in others) if others else True
            if asp>=self.SV_ASPECT_CHG: ctype='vehicle_flip'
            elif is_moto and alone:       ctype='motorcycle_fall'
            elif alone:                   ctype='single_vehicle_anomaly'
            else: continue
            return self._make_ev(frame_num,ctype,[v],vehicles,
                                  extra={'decel_bhs2':round(dv,3),
                                         'aspect_change':round(asp,3),'alone':alone})

        # ── D: static impact ──────────────────────────────
        for v in vehicles:
            if not self._ok(v['id'],frame_num): continue
            if v['speed_bhps']<self.SI_MIN_SPEED: continue
            dv=self.tracker.decel_bh_s2(v['id'])
            if dv<self.SI_DECEL: continue
            others=[u for u in vehicles if u['id']!=v['id']]
            if any(center_dist(v['bbox'],u['bbox'])/v['bbox_height']<self.SI_ALONE_BH
                    for u in others): continue
            return self._make_ev(frame_num,'static_impact',[v],vehicles,
                                  extra={'decel_bhs2':round(dv,3)})
        return None

print("FullCollisionDetector v3 ready (BH/s² thresholds)")
print()
print("Thresholds:")
d=FullCollisionDetector
print(f"  Vehicle-vehicle : >{d.VV_DECEL} BH/s²  (proximity: <{d.VV_DIST_BH} BH apart)")
print(f"  Single vehicle  : >{d.SV_DECEL} BH/s²  + aspect change OR alone")
print(f"  Ped decel       : >{d.PS_VEH_DECEL} BH/s²  OR ped spike >{d.PS_PED_SPIKE} BH/s")
print(f"  Static impact   : >{d.SI_DECEL} BH/s²  alone")
print()
print("Car stops gently from 10 km/h (was 89% percentage flag):")
print("  speed_bhps ≈ 0.4 BH/s → decel ≈ 1.3 BH/s² → NOT flagged ✅")

FullCollisionDetector v3 ready (BH/s² thresholds)

Thresholds:
  Vehicle-vehicle : >3.5 BH/s²  (proximity: <2.0 BH apart)
  Single vehicle  : >4.0 BH/s²  + aspect change OR alone
  Ped decel       : >2.5 BH/s²  OR ped spike >2.0 BH/s
  Static impact   : >5.0 BH/s²  alone

Car stops gently from 10 km/h (was 89% percentage flag):
  speed_bhps ≈ 0.4 BH/s → decel ≈ 1.3 BH/s² → NOT flagged ✅


In [6]:
# 3.3 Diagnostic — confirm vehicle detection + measure actual values
# Checks: (1) YOLO sees vehicles, (2) speeds are non-zero, (3) IoU at crash moment

import numpy as np

vids = sorted(VIDEOS_DIR.glob('*.mp4'))
print(f"Videos available: {len(vids)}")
if not vids:
    print("ERROR: No videos found. Check VIDEOS_DIR path.")
else:
    # Quick detection check on first video (first 10 seconds)
    vid = vids[0]
    cap = cv2.VideoCapture(str(vid))
    fps_v = cap.get(cv2.CAP_PROP_FPS) or 30.0
    limit = min(int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), int(10*fps_v))
    print(f"\nChecking {vid.name} (first 10s @ {fps_v:.0f}fps)")

    trk = VehicleTracker(fps=fps_v)
    counts=[]; max_iou=0.0; max_spd=0.0; max_decel=0.0
    iou_events=[]

    for fn in range(limit):
        ret,frame=cap.read()
        if not ret: break
        vehicles,peds=trk.update(frame,fn)
        counts.append(len(vehicles))
        for v in vehicles:
            max_spd=max(max_spd, v['speed_bhps'])
        # Check all pairs for IoU
        for i in range(len(vehicles)):
            for j in range(i+1,len(vehicles)):
                iou=box_iou(vehicles[i]['bbox'],vehicles[j]['bbox'])
                dist_bh=center_dist(vehicles[i]['bbox'],vehicles[j]['bbox'])/max(vehicles[i]['bbox_height'],1)
                da=trk.decel_bh_s2(vehicles[i]['id'])
                db=trk.decel_bh_s2(vehicles[j]['id'])
                if iou>max_iou: max_iou=iou
                if max(da,db)>max_decel: max_decel=max(da,db)
                if iou>0.01 or dist_bh<3.0:
                    iou_events.append({'fn':fn,'iou':round(iou,4),'dist_bh':round(dist_bh,2),
                                        'decel':round(max(da,db),3)})
    cap.release()

    print(f"\n  Vehicle detection:")
    print(f"    Frames checked        : {len(counts)}")
    print(f"    Avg vehicles/frame    : {sum(counts)/max(len(counts),1):.1f}")
    print(f"    Max vehicles in frame : {max(counts) if counts else 0}")
    print(f"    Frames with 0 vehicles: {counts.count(0)}")
    print(f"\n  Physics values seen:")
    print(f"    Max speed_bhps        : {max_spd:.3f} BH/s")
    print(f"    Max decel seen        : {max_decel:.3f} BH/s²")
    print(f"    Max IoU between pairs : {max_iou:.4f}")
    print(f"    Pairs within 3 BH     : {len(iou_events)}")
    if iou_events:
        top=sorted(iou_events,key=lambda x:-x['iou'])[:5]
        print(f"\n  Top 5 closest pairs:")
        print(f"    {'frame':>6} {'iou':>7} {'dist_bh':>8} {'decel':>7}")
        for e in top:
            print(f"    {e['fn']:>6} {e['iou']:>7.4f} {e['dist_bh']:>8.2f} {e['decel']:>7.3f}")
    print(f"\n  Current thresholds:")
    print(f"    VV_IOU    : {FullCollisionDetector.VV_IOU}  (IoU contact — now 0.02)")
    print(f"    VV_DIST_BH: {FullCollisionDetector.VV_DIST_BH}  (proximity in BH)")
    print(f"    VV_DECEL  : {FullCollisionDetector.VV_DECEL}  BH/s²")
    if max(counts,default=0)==0:
        print("\n  ⚠ YOLO detected NO vehicles — check model loaded correctly (cell 2.1)")
    elif max_iou < 0.02:
        print(f"\n  No direct vehicle contact in first 10s (max IoU={max_iou:.4f})")
        print(f"  This is normal — crash happens at a specific moment in the video.")
        print(f"  New A1 trigger (iou>=0.02) will catch it when it occurs.")
    else:
        print(f"\n  ✅ IoU events detected — new A1 trigger will flag these")


Videos available: 230

Checking -GpvLzopst8.mp4 (first 10s @ 30fps)

  Vehicle detection:
    Frames checked        : 300
    Avg vehicles/frame    : 6.3
    Max vehicles in frame : 11
    Frames with 0 vehicles: 0

  Physics values seen:
    Max speed_bhps        : 5.417 BH/s
    Max decel seen        : 0.000 BH/s²
    Max IoU between pairs : 0.9672
    Pairs within 3 BH     : 3085

  Top 5 closest pairs:
     frame     iou  dist_bh   decel
        61  0.9672     0.00   0.000
        62  0.9672     0.00   0.000
        64  0.9461     0.01   0.000
        11  0.9315     0.03   0.000
        10  0.9308     0.01   0.000

  Current thresholds:
    VV_IOU    : 0.05  (IoU contact — now 0.02)
    VV_DIST_BH: 2.0  (proximity in BH)
    VV_DECEL  : 3.5  BH/s²

  ✅ IoU events detected — new A1 trigger will flag these


In [7]:
# 4.1 Full video pipeline
def process_video(video_path, max_sec=120, visualize=True):
    video_path = Path(video_path)
    cap   = cv2.VideoCapture(str(video_path))
    fps   = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    limit = min(total, int(max_sec * fps))
    h_f   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    w_f   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))

    tracker  = VehicleTracker(fps=fps)
    detector = FullCollisionDetector(fps=fps, tracker=tracker)
    collisions = []; timeline = []; snap = None

    bar = tqdm(
        total=limit,
        desc=f"{video_path.name[:28]}",
        dynamic_ncols=True,
        leave=True,
        bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}] {postfix}'
    )

    for fn in range(limit):
        ret, frame = cap.read()
        if not ret: break

        vehicles, peds = tracker.update(frame, fn)
        all_tracks     = vehicles + peds
        event          = detector.check(vehicles, peds, fn)
        max_bh         = max((t['speed_bhps'] for t in vehicles), default=0)

        timeline.append({'frame': fn, 'time': fn/fps,
                          'n_v': len(vehicles), 'max_bhps': max_bh})

        if event:
            collisions.append(event)
            if snap is None: snap = frame.copy()
            vb     = event['vehicle_b']
            vb_str = f"+ {vb['class']}(ID{vb['id']})" if vb else "(single vehicle)"
            tqdm.write(
                f"  🚨 [{event['collision_type']:<22}] "
                f"t={event['time_sec']:.1f}s | "
                f"{event['vehicle_a']['class']}(ID{event['vehicle_a']['id']}) {vb_str} | "
                f"decel={event['decel_a_bhs2']:.2f} BH/s²"
            )

        # Update bar every 30 frames to avoid slowdown
        if fn % 30 == 0:
            bar.set_postfix(
                vehicles=len(vehicles),
                events=len(collisions),
                spd=f"{max_bh:.1f}BH/s",
                refresh=False
            )
        bar.update(1)

    bar.set_postfix(vehicles='—', events=len(collisions), spd='—')
    bar.close()
    cap.release()

    print(f"  Total events: {len(collisions)}")

    if visualize and collisions and snap is not None:
        ev = collisions[0]
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        axes[0].imshow(cv2.cvtColor(snap, cv2.COLOR_BGR2RGB))
        for t in ev['all_tracks']:
            x1, y1, x2, y2 = t['bbox']
            inv = t['id'] in (
                [ev['vehicle_a']['id']] +
                ([ev['vehicle_b']['id']] if ev['vehicle_b'] else []))
            c  = '#ff2222' if inv else '#3498db'
            lw = 3 if inv else 1
            axes[0].add_patch(patches.Rectangle(
                (x1, y1), x2-x1, y2-y1,
                linewidth=lw, edgecolor=c, facecolor='none'))
            axes[0].text(x1, y1-4, f"ID{t['id']} {t['speed_bhps']:.1f}BH/s",
                color='white', fontsize=7,
                bbox=dict(facecolor=c, alpha=0.7, pad=1, edgecolor='none'))
        pb   = ev['collision_bbox_padded']
        pb_c = [max(0,pb[0]), max(0,pb[1]), min(w_f,pb[2]), min(h_f,pb[3])]
        axes[0].add_patch(patches.Rectangle(
            (pb_c[0], pb_c[1]), pb_c[2]-pb_c[0], pb_c[3]-pb_c[1],
            linewidth=2, edgecolor='#ffff00', facecolor='none',
            linestyle='--', label='VLM crop (20% padded)'))
        axes[0].set_title(
            f"[{ev['collision_type']}] at {ev['time_sec']:.1f}s | yellow = VLM crop")
        axes[0].axis('off')

        axes[1].plot([t['time'] for t in timeline],
                      [t['max_bhps'] for t in timeline],
                      'b-', lw=1.5, label='max speed (BH/s)')
        for ev2 in collisions:
            axes[1].axvline(ev2['time_sec'], color='red', ls='--',
                             lw=1.5, alpha=0.7, label=ev2['collision_type'])
        axes[1].set_xlabel('Time (s)'); axes[1].set_ylabel('Max speed (BH/s)')
        axes[1].set_title('Scale-invariant speed timeline')
        axes[1].grid(True, alpha=0.3)
        handles, labels = axes[1].get_legend_handles_labels()
        seen = set(); uhan = []; ulab = []
        for h_, l_ in zip(handles, labels):
            if l_ not in seen: seen.add(l_); uhan.append(h_); ulab.append(l_)
        axes[1].legend(uhan, ulab, fontsize=8)
        plt.tight_layout(); plt.show()

    return collisions, timeline, snap


vids = sorted(VIDEOS_DIR.glob('*.mp4'))
if vids:
    events, tl, snap = process_video(vids[0])
else:
    print("No videos found")

-GpvLzopst8.mp4: 100%|██████████| 1830/1830 [00:28<00:00] , events=0, spd=—, vehicles=—

  Total events: 0


In [ ]:
# 4.2 Batch evaluation
def run_batch(vids, n=20):
    results=[]
    for vf in tqdm(vids[:n], desc="Batch", position=0, leave=True, dynamic_ncols=True,
                   bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]'):
        try:
            evs,tl,_=process_video(vf,visualize=False)
            n_ev=len(evs)
            types=[e['collision_type'] for e in evs]
            # Single summary line per video
            tqdm.write(f"  {vf.name[:35]:<35} events={n_ev}  {types if types else ''}")
            results.append({'video':vf.name,'n_events':n_ev,'types':types})
        except Exception as ex:
            tqdm.write(f"  ERROR {vf.name}: {ex}")
    return results

vids=sorted(VIDEOS_DIR.glob('*.mp4'))
print(f"Running batch on first 20 of {len(vids)} videos...")
batch=run_batch(vids,20)
from collections import Counter
all_types=Counter(t for r in batch for t in r['types'])
print(f"  Videos with events : {sum(1 for r in batch if r['n_events']>0)}/{len(batch)}")
print(f"  Events by type     : {dict(all_types)}")
out=RESULTS_DIR/'bytetrack_v3_results.json'
with open(out,'w') as f: json.dump(batch,f,indent=2)
print(f"Saved: {out}")

Running batch on first 20 of 230 videos...


Batch:   5%|▌         | 1/20 [00:27<08:45]

  Total events: 0
  -GpvLzopst8.mp4                     events=0  



Batch:  10%|█         | 2/20 [01:10<10:54]

  Total events: 0
  -UQZMlA_xEs.mp4                     events=0  


...

  Total events: 0
  3Xd6PxEvbNk.mp4                     events=0  



Batch:  95%|█████████▌| 19/20 [11:00<00:38]

  Total events: 0
  3h_zF5nLVy4.mp4                     events=0  



Batch: 100%|██████████| 20/20 [11:13<00:00]


  Total events: 0
  3p_-1RjvCmw.mp4                     events=0  
  Videos with events : 0/20
  Events by type     : {}
Saved: /content/drive/MyDrive/Colab Notebooks/grad-project/program-memory/tracking_results/bytetrack_v3_results.json


In [ ]:
# 5.1 VLM integration export
def extract_accident_context(event, frame_bgr, frame_w, frame_h):
    """
    Package event for VLM_Triage_Qwen25_v2.ipynb.
    Adds physics data from the BH/s² tracker.
    collision_bbox_padded already has 20% padding for VLM crop.
    """
    inv=[event['vehicle_a']] + ([event['vehicle_b']] if event['vehicle_b'] else [])
    pb=event.get('collision_bbox_padded',event['collision_bbox'])
    pb_clamped=[max(0,pb[0]),max(0,pb[1]),min(frame_w,pb[2]),min(frame_h,pb[3])]
    return {
        'involved_vehicles':  inv,
        'vehicle_types':      list({v['class'] for v in inv}),
        'vehicle_count':      len(inv),
        'pedestrian_count':   sum(1 for t in event['all_tracks'] if t['class']=='person'),
        'max_vehicle_overlap': event['iou'],
        'collision_point':    ((inv[0]['cx']+(inv[1]['cx'] if len(inv)>1 else inv[0]['cx']))/2,
                               (inv[0]['cy']+(inv[1]['cy'] if len(inv)>1 else inv[0]['cy']))/2),
        'accident_bbox':      pb_clamped,   # 20%-padded, clamped to frame
        'all_vehicles_count': len(event['all_tracks']),
        'collision_type':     event['collision_type'],
        'detections':         event['all_tracks'],
        '_physics': {
            'decel_a_bhs2':    event['decel_a_bhs2'],
            'decel_b_bhs2':    event['decel_b_bhs2'],
            'speed_a_bhps':    event['vehicle_a']['speed_bhps'],
            'speed_b_bhps':    event['vehicle_b']['speed_bhps'] if event['vehicle_b'] else 0.0,
            'direct_contact':  event['iou'] > 0.05,
        }
    }

if 'events' in dir() and events and snap is not None:
    h_f,w_f=snap.shape[:2]
    ctx=extract_accident_context(events[0],snap,w_f,h_f)
    print("VLM context ready:")
    print(f"  Collision type   : {ctx['collision_type']}")
    print(f"  Vehicles         : {ctx['vehicle_types']}")
    print(f"  Padded bbox      : {ctx['accident_bbox']}")
    print(f"  Physics          : {ctx['_physics']}")
    print()
    print("Pass this as yolo_data to generate_triage_report() in VLM notebook.")
